# MGS-1 Reduced-Mass Geotechnical Index Testing

This notebook reproduces the calculation and plotting workflow used for the thesis figures and tables.
It is intentionally streamlined for review and reuse: each section loads or defines the input data, performs
the relevant calculations, and exports only the figures/tables used in the thesis.


---
## 0. Setup


In [ ]:
import re
import statistics as stats
from pathlib import Path

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
TABLE_DIR = Path("tables")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from openpyxl import load_workbook


### 0.1 Figure Style
Shared Matplotlib defaults for non-PSD figures. PSD figures use the dedicated `PLOT_CFG` settings below.


In [ ]:
# ── Global figure style ─────────────────────────────────────────────────────
STYLE_CFG = {
    "figsize"     : (12, 8),
    "dpi_screen"  : 120,
    "dpi_save"    : 300,
    "font_size"   : 14,
    "title_size"  : 12,
    "label_size"  : 14,
    "legend_size" : 12,
    "linewidth"   : 2.0,
    "markersize"  : 8,
    "grid_alpha"  : 0.35,
}

def apply_style(cfg=None):
    c = STYLE_CFG if cfg is None else cfg
    plt.rcParams.update({
        "figure.figsize"  : c["figsize"],
        "figure.dpi"      : c["dpi_screen"],
        "savefig.dpi"     : c["dpi_save"],
        "font.size"       : c["font_size"],
        "axes.titlesize"  : c["title_size"],
        "axes.labelsize"  : c["label_size"],
        "legend.fontsize" : c["legend_size"],
        "lines.linewidth" : c["linewidth"],
        "lines.markersize": c["markersize"],
        "axes.grid"       : True,
        "grid.linewidth"  : 0.35,
        "axes.spines.top"   : True,
        "axes.spines.right" : True,
        "axes.spines.left"  : True,
        "axes.spines.bottom": True,
    })

apply_style()

OUT = Path("figures")
WORD_OUT = Path("figures_word/png_600dpi")
OUT.mkdir(exist_ok=True)
WORD_OUT.mkdir(parents=True, exist_ok=True)

WORD_DPI = 600

def savefig(fig, name, fmt="pdf"):
    """Save thesis PDF and a high-resolution PNG for Word/PDF export on macOS."""
    pdf_path = OUT / f"{name}.pdf"
    word_png_path = WORD_OUT / f"{name}.png"
    fig.savefig(pdf_path, format="pdf", dpi=STYLE_CFG["dpi_save"], bbox_inches="tight")
    fig.savefig(word_png_path, format="png", dpi=WORD_DPI, bbox_inches="tight", facecolor="white")
    print(f"Saved: {pdf_path}; Word PNG: {word_png_path}")

G = 9.81  # m/s²

# Keep notebook execution quiet when run as a script.
try:
    display
except NameError:
    def display(*args, **kwargs):
        pass


### 0.2 Mass-Centric Visual Style
A single colour/marker convention is used for each nominal specimen mass across the notebook.


In [ ]:
# ── Consistent mass-centric style (B/W-readable) ──────────────────────────────
# Design principle (separates four dimensions while remaining B/W readable):
#   • Mass    -> unique marker shape + colour, kept consistent across all figures
#   • Method -> fill style (dry=filled, wet/hydro=open) + a subtle tint for wet
#   • Replicate -> line style (1=solid, 2=dashed, 3=dotted)
#   • General -> colour for print, still readable in B/W through marker shape and fill
BW_MODE = False   # True for black-and-white version (colours converted to greyscale)

# Marker per specimen mass (distinct shapes for B/W readability)
MASS_MARKER = {
    100: "o",   # circle
    50 : "s",   # square
    10 : "^",   # upward triangle
    2  : "D",   # diamond
    0.5: "P",   # filled plus
}

# Colours per specimen mass
MASS_COLOR_RGB = {
    100:  "tab:purple",
    50:   "tab:blue",
    10:   "tab:orange",
    2:    "tab:green",
    0.5:  "tab:red",
}
# Greyscale palette for B/W mode (dark to light from large to small mass)
MASS_COLOR_BW = {
    100:  "#000000",
    50:   "#2b2b2b",
    10:   "#5a5a5a",
    2:    "#8a8a8a",
    0.5:  "#b5b5b5",
}

# Method coding
# Fill style: dry = filled, wet + hydro = open
METHOD_FILL = {
    "dry"         : "hollow",
    "wet"         : "filled",
    "hydro_bulk"  : "filled",   # bulk hydrometer is primary: filled, with its own line style
    "hydro_fines" : "hollow",
    "stitched"    : "filled",
}

# Line width per method (dashed/dotted lines look visually thicker at the same lw).
# Reduced after review: sieve/hydrometer signal and stitched median
# were too thick; now more proportional for detailed PSD figures.
METHOD_LW = {
    "dry"         : 1.4,   # dry sieve
    "wet"         : 1.4,   # wet sieve
    "hydro_bulk"  : 1.4,   # bulk-hydrometer
    "hydro_fines" : 0.6,   # fines-only hydro - intentionally thinner (support/reference)
    "stitched"    : 1.4,   # per-mass and median stitched - thinner, readable
}

# Color tint per method (0 = unchanged, 1 = white). Keep a subtle tint for wet.
METHOD_TINT = {
    "dry"         : 0.00,
    "wet"         : 0.25,    # subtle lighter tone so wet is not identical to dry
    "hydro_bulk"  : 0.00,
    "hydro_fines" : 0.20,
    "stitched"    : 0.00,
}

# Line style per replicate (applies to all methods). Used when REPLICATE_LS_ENABLED=True;
# otherwise the method controls the line style (useful when the same linestyle is desired for all
# replicates of the same method).
REPLICATE_LS_ENABLED = True
REPLICATE_LS = {
    0: "-",                   # solid
    1: (0, (6, 2)),           # long dashed
    2: (0, (1, 1.5)),         # tight dotted
    3: (0, (4, 1, 1, 1)),     # dash-dot
}

# Fallback line style per method (only used when REPLICATE_LS_ENABLED=False).
METHOD_LS_FALLBACK = {
    "dry"         : "-",
    "wet"         : (0, (6, 2)),
    "hydro_bulk"  : (0, (1, 1.2)),
    "hydro_fines" : (0, (4, 1, 1, 1)),
    "stitched"    : "-",
}

def _nearest_mass(mass):
    """Snap an arbitrary mass to the closest nominal group (log-distance).
    Used so 55 g (loose/tapped data) maps to the 50 g style, etc.
    """
    if mass in MASS_MARKER:
        return mass
    keys = list(MASS_MARKER.keys())
    return min(keys, key=lambda k: abs(np.log10(k) - np.log10(mass)))

def mass_color(mass):
    """Return color for a given specimen mass (respects BW_MODE)."""
    m = _nearest_mass(mass)
    return (MASS_COLOR_BW if BW_MODE else MASS_COLOR_RGB)[m]

def mass_marker(mass):
    """Return marker for a given specimen mass."""
    return MASS_MARKER[_nearest_mass(mass)]

def _tint(col, factor):
    """Blend RGB color toward white. factor=0 → unchanged, factor=1 → white."""
    from matplotlib.colors import to_rgb
    r, g, b = to_rgb(col)
    return (r + (1-r)*factor, g + (1-g)*factor, b + (1-b)*factor)

def style_for(mass, method="dry", replicate_idx=0, **overrides):
    """Return a style dict suitable for ax.plot / ax.errorbar.
    Style dimensions:
      • color/marker - per mass
      • line style   - per replicate when REPLICATE_LS_ENABLED, otherwise per method
      • fill style   - per method (dry=filled, wet/hydro_fines=open, hydro_bulk=filled)
      • line width   - per method
      • color tint   - per method (subtle for wet/hydro_fines)
    Mass can be any value; it is snapped to the nearest nominal group.
    """
    base_col = mass_color(mass)
    col = _tint(base_col, METHOD_TINT.get(method, 0.0))
    mk  = mass_marker(mass)

    # Fyllstil
    fill_kind = METHOD_FILL.get(method, "filled")
    mfc = col if fill_kind == "filled" else "white"

    # Linjestil
    if REPLICATE_LS_ENABLED:
        ls = REPLICATE_LS.get(replicate_idx, "-")
    else:
        ls = METHOD_LS_FALLBACK.get(method, "-")

    lw = METHOD_LW.get(method, 2.0)

    st = dict(color=col, marker=mk, linestyle=ls,
              markerfacecolor=mfc, markeredgecolor=col,
              markeredgewidth=1.4, linewidth=lw, markersize=6.5)
    st.update(overrides)
    return st

def parse_mass(tid):
    """Extract nominal specimen mass from test ID (e.g. '50A', 'F10', '2W1' → 50/10/2)."""
    s = str(tid).replace(",", ".")
    m = re.match(r"^F?(\d+(?:\.\d+)?)", s)
    return float(m.group(1)) if m else np.nan

def parse_rep_idx(tid):
    """Replicate index from suffix (A/W1 → 0, B/W2 → 1, W3 → 2)."""
    s = str(tid)
    if s.endswith("W1") or s.endswith("A"):
        return 0
    if s.endswith("W2") or s.endswith("B"):
        return 1
    if s.endswith("W3") or s.endswith("C"):
        return 2
    return 0

print("Style loaded. BW_MODE =", BW_MODE)


---
## 1. Water Content (ASTM D2216)


### 1.1 Input Data


In [ ]:
# Figure/Table input: raw oven-drying measurements.
# Columns: test_id, container_g, container_wet_g, container_dry_g.
wc_raw = pd.read_csv(DATA_DIR / "water_content_raw.csv")
WC_RUNS = [
    (f"MGS1_{r.test_id}", r.container_g, r.container_wet_g, r.container_dry_g)
    for r in wc_raw.itertuples(index=False)
]


### 1.2 Calculations and Summary Statistics


In [ ]:
def wc_compute(Mc, Mcws, Mcs):
    """Compute water content w [%] per ASTM D2216."""
    Mw = Mcws - Mcs
    Ms = Mcs - Mc
    return (Mw / Ms) * 100 if Ms > 0 else np.nan

def wc_mass_from_label(label):
    """Parse nominal mass from test label."""
    for size in [100, 10, 2, 0.5]:
        if str(size) in label:
            return float(size)
    return None

# Compute water content for all specimens
wc_results = []
for label, Mc, Mcws, Mcs in WC_RUNS:
    w = wc_compute(Mc, Mcws, Mcs)
    wc_results.append({
        "label": label,
        "mass_g": wc_mass_from_label(label),
        "Mdry_g": Mcs - Mc,
        "w_pct": w,
    })

# Group by mass level and compute statistics
wc_by_mass = {}
for r in wc_results:
    wc_by_mass.setdefault(r["mass_g"], []).append(r["w_pct"])

wc_levels  = sorted(wc_by_mass.keys())
wc_median  = {L: stats.median(v) for L, v in wc_by_mass.items()}
wc_sd      = {L: stats.stdev(v) if len(v) > 1 else np.nan for L, v in wc_by_mass.items()}
wc_ref     = wc_median[max(wc_levels)]   # 100 g median as reference

print(f"{'Label':<18} {'w [%]':>8} {'Dry mass [g]':>14}")
print("-" * 43)
for r in wc_results:
    print(f"{r['label']:<18} {r['w_pct']:>8.3f} {r['Mdry_g']:>14.3f}")

print(f"\nSummary (median ± SD):")
print(f"{'Mass [g]':<12} {'n':>4} {'Median w [%]':>14} {'SD [%]':>10} {'Δw vs 100 g':>14}")
print("-" * 56)
for L in wc_levels:
    sd = wc_sd[L]
    delta = wc_median[L] - wc_ref
    print(f"{L:<12g} {len(wc_by_mass[L]):>4} {wc_median[L]:>14.3f} "
          f"{sd if np.isfinite(sd) else float('nan'):>10.3f} {delta:>+14.3f}")


### 1.3 Thesis Figure


In [ ]:
# Figure 3.1 — Individual water-content replicates with group median
x    = np.array(wc_levels)
y    = np.array([wc_median[L] for L in wc_levels])
yerr = np.array([wc_sd[L] for L in wc_levels])
rng = np.random.default_rng(42)

fig2_wc, ax = plt.subplots()
xs_all = np.array([r["mass_g"] for r in wc_results], float)
ys_all = np.array([r["w_pct"] for r in wc_results], float)
xs_jit = 10 ** (np.log10(xs_all) + rng.normal(0, 0.025, len(xs_all)))

ax.scatter(xs_jit, ys_all, s=50, color="darkorange", alpha=0.9,
           label="Replicates", zorder=3)
ax.plot(x, y, "o--", color="tab:blue", lw=1.8, ms=5,
        label="Group median", zorder=4)
ax.axhline(wc_ref, color="gray", ls="--", lw=1.2,
           label=f"100 g reference: {wc_ref:.2f} %")

ax.set_xscale("log")
ax.set_xticks(wc_levels)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlabel("Specimen mass [g]")
ax.set_ylabel("Water content, w [%]")
ax.set_ylim(0, 10)
ax.legend(frameon=False)
plt.tight_layout()
savefig(fig2_wc, "fig_3_1_water_content")
plt.show()


---
## 2. Particle Density by Water Pycnometer (ASTM D854)


### 2.1 Load and Clean Pycnometer Data


In [ ]:
# Figure/Table input: processed pycnometer results.
df_pyk = pd.read_csv(DATA_DIR / "pycnometer_processed.csv").rename(columns={
    "temperature_C": "temp_C",
    "rho_s_g_cm3": "rho_s",
    "mass_g": "ms_g",
})
df_pyk["pyk_id"] = np.nan
df_pyk["fraction"] = (
    df_pyk["fraction"].astype(str)
    .str.replace(">75 um", ">75 µm", regex=False)
    .str.replace("<75 um", "<75 µm", regex=False)
)
for c in ["ms_g", "temp_C", "rho_s", "Gs"]:
    df_pyk[c] = pd.to_numeric(df_pyk[c], errors="coerce")

def nominal_mass_from_test_id(test_id):
    match = re.match(r"^(\d+(?:\.\d+)?)", str(test_id).strip().replace(",", "."))
    return float(match.group(1)) if match else np.nan

df_pyk["mass_g"] = df_pyk["test_id"].map(nominal_mass_from_test_id)
df_pyk["mass_label"] = df_pyk["mass_g"].apply(
    lambda v: f"{int(v)} g" if not pd.isna(v) and float(v).is_integer() else f"{v:g} g"
)
frac_order = {"Bulk": 0, ">75 µm": 1, "<75 µm": 2}
df_pyk = (
    df_pyk.assign(_fraction_order=df_pyk["fraction"].map(frac_order).fillna(99))
    .sort_values(["_fraction_order", "mass_g", "test_id"])
    .drop(columns="_fraction_order")
    .reset_index(drop=True)
)

display(df_pyk[["test_id", "mass_label", "fraction", "ms_g", "temp_C", "rho_s", "Gs"]])


### 2.2 Summary Tables


In [ ]:
# ── Bulk summary table ───────────────────────────────────────────────────────
bulk = df_pyk[df_pyk["fraction"] == "Bulk"].copy()

bulk_summary = (
    bulk.groupby("mass_g", as_index=False)
    .agg(n=("rho_s","count"), med_rho=("rho_s","median"),
         sd_rho=("rho_s","std"),  med_Gs=("Gs","median"), sd_Gs=("Gs","std"))
    .sort_values("mass_g")
)

ref_mass_pyk = bulk_summary["mass_g"].max()
ref_rho_pyk  = bulk_summary.loc[bulk_summary["mass_g"] == ref_mass_pyk, "med_rho"].iloc[0]
bulk_summary["delta_rho"] = (bulk_summary["med_rho"] - ref_rho_pyk).round(4)
bulk_summary["mass_label"] = bulk_summary["mass_g"].apply(
    lambda v: f"{int(v)} g" if float(v).is_integer() else f"{v:g} g"
)

print("Bulk summary (median-based):")
display(bulk_summary[["mass_label","n","med_rho","sd_rho","delta_rho","med_Gs","sd_Gs"]].rename(columns={
    "mass_label": "Specimen mass", "n": "n",
    "med_rho": "Median ρs [g/cm³]", "sd_rho": "SD ρs [g/cm³]",
    "delta_rho": "Δρs vs reference [g/cm³]",
    "med_Gs": "Median Gs [-]", "sd_Gs": "SD Gs [-]",
}).round(4))


### 2.3 Thesis Figures


In [ ]:
# ── Figure 1: Replicates + group median (bulk) ───────────────────────────────
fig1_pyk, ax = plt.subplots()
ax.scatter(bulk["mass_g"], bulk["rho_s"], s=70, label="Replicates", zorder=3)
ax.plot(bulk_summary["mass_g"], bulk_summary["med_rho"],
        "o--", lw=1.8, ms=5, label="Group median", zorder=4)
ax.axhline(ref_rho_pyk, ls="--", lw=1.2, alpha=0.8,
           label=f"Reference median ({ref_mass_pyk:g} g): {ref_rho_pyk:.4f} g/cm³")

ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlabel("Specimen mass [g]")
ax.set_ylabel("Particle density, ρs [g/cm³]")
ax.legend(frameon=False)
plt.tight_layout()
savefig(fig1_pyk, "fig_3_2_rho_s_bulk")
plt.show()


> Figure 3.3 uses a black-and-white-readable style: each fraction has a distinct marker and line style in addition to colour.


In [ ]:
# Fraction style shared by Figure 3.3.
FRACTION_STYLE = {
    "Bulk"   : {"marker": "o", "color": "tab:blue", "ls": "-",
                 "label": "Bulk"},
    ">75 µm" : {"marker": "s", "color": "tab:orange", "ls": (0,(6,2)),
                 "label": ">75 µm"},
    "<75 µm" : {"marker": "^", "color": "tab:green", "ls": (0,(1,1.5)),
                 "label": "<75 µm"},
}

def _frac_key(name):
    s = str(name).strip().lower()
    if "bulk" in s:
        return "Bulk"
    if ">75" in s or "over" in s or "coarse" in s:
        return ">75 µm"
    if "<75" in s or "under" in s or "fines" in s:
        return "<75 µm"
    return None


In [ ]:
# ── Figure 3.3: all fractions + bulk replicates (B/W-readable) ────────────────
# Medians for all fractions + individual bulk replicate points. Bulk replicates are
# Shown as open circles so they are easy to distinguish from the median marker.
summary_all_pyk = (
    df_pyk.groupby(["fraction","mass_g"], as_index=False)
    .agg(n=("rho_s","count"), med_rho=("rho_s","median"), sd_rho=("rho_s","std"))
)

fig5_pyk, ax = plt.subplots()

for frac, sub in summary_all_pyk.groupby("fraction"):
    sub = sub.sort_values("mass_g")
    key = _frac_key(frac)
    st  = FRACTION_STYLE.get(key, {"marker":"D","color":"0.3","ls":":","label":str(frac)})
    ax.plot(sub["mass_g"], sub["med_rho"],
            marker=st["marker"], ls=st["ls"], color=st["color"],
            markerfacecolor=st["color"], markeredgecolor=st["color"],
            markersize=7, lw=1.8, label=f"{st['label']} median")

# Bulk replicates (small open circles in the bulk colour)
bulk_rep = df_pyk[df_pyk["fraction"].str.strip().str.lower() == "bulk"].sort_values("mass_g")
bulk_col = FRACTION_STYLE["Bulk"]["color"]
ax.scatter(bulk_rep["mass_g"], bulk_rep["rho_s"],
           marker="o", s=38, facecolor="white",
           edgecolor=bulk_col, linewidth=1.2,
           alpha=0.85, label="Bulk replicates", zorder=2)

ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlabel("Specimen mass [g]")
ax.set_ylabel("Particle density, ρs [g/cm³]")
ax.legend(frameon=False)
plt.tight_layout()
savefig(fig5_pyk, "fig_3_3_rho_s_fractions")
plt.show()


### Figure 4.1 - Mineralogical Plausibility Check
This discussion figure belongs with the particle-density results and is therefore placed here.


In [ ]:
# Figure 4.1 - Mineralogical plausibility of bulk particle density.
RHO_MIN_LOWER = 2.88
RHO_MIN_MEAN  = 3.04
RHO_MIN_UPPER = 3.16

literature_refs = [
    ("MGS-1\n(gas pyk.)",           2.89, None, "Karacasulu et al. 2025"),
    ("InSight regolith\n(assumed)", 2.80, None, "Morgan et al. 2018"),
]

YLIM = (1.5, 3.5)
n_meas = len(bulk_summary)
n_lit  = len(literature_refs)
GAP = 0.7

x_meas = np.arange(n_meas, dtype=float)
x_lit  = np.arange(n_lit, dtype=float) + n_meas + GAP

fig6_pyk, ax = plt.subplots(figsize=(9, 5))

ax.axhspan(RHO_MIN_LOWER, RHO_MIN_UPPER, color="tab:green", alpha=0.12,
           label=(f"Mineralogical plausibility range\n"
                  f"(lower {RHO_MIN_LOWER}, mean {RHO_MIN_MEAN}, "
                  f"upper {RHO_MIN_UPPER} g/cm³)"))
ax.axhline(RHO_MIN_MEAN, color="tab:green", ls="--", lw=1.2, alpha=0.7)

ax.errorbar(
    x_meas, bulk_summary["med_rho"], yerr=bulk_summary["sd_rho"],
    fmt="o", markersize=6, color="tab:blue",
    capsize=5, lw=1.2, zorder=4,
    label="Median ± 1 SD (this study)"
)

for xi, (_label, val, err, _source) in zip(x_lit, literature_refs):
    if err is not None:
        ax.errorbar(xi, val, yerr=err, fmt="o",
                    color="tab:orange", markersize=9,
                    capsize=5, lw=1.2, zorder=3)
    else:
        ax.scatter(xi, val, marker="o", s=40, color="tab:orange", zorder=3)
ax.scatter([], [], marker="o", s=40, color="tab:orange",
           label="Reference value (literature)")

meas_labels = list(bulk_summary["mass_label"])
lit_labels = [label for (label, _, _, _) in literature_refs]
ax.set_xticks(list(x_meas) + list(x_lit))
ax.set_xticklabels(meas_labels + lit_labels, fontsize=9)

ax.set_ylim(*YLIM)
ax.set_ylabel("Particle density, ρs [g/cm³]")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="lower right", frameon=False, fontsize=9)

plt.tight_layout()
savefig(fig6_pyk, "fig_4_1_rho_s_plausibility")
plt.show()


In [ ]:
STYLE_CFG = {
    "figsize"     : (12, 8),
    "dpi_screen"  : 120,
    "dpi_save"    : 300,
    "font_size"   : 14,
    "title_size"  : 12,
    "label_size"  : 14,
    "legend_size" : 10,
    "linewidth"   : 2.0,
    "markersize"  : 8,
    "grid_alpha"  : 0.35,
}


---
## 3. Particle-Size Distribution (ASTM D6913 + ASTM D7928)


In [ ]:
# ── PSD configuration ───────────────────────────────────────────────────────
PLOT_CFG = {
    "xlim_full"           : (0.001, 8.0),
    "xlim_sieve"          : (0.04,  8.0),
    "xlim_hydro"          : (0.001, 0.075),
    "ylim"                : (0, 105),
    # Default PSD figure size. (14, 8) is a good compromise: tall enough to read
    # the USCS header, fraction names, and legend without stretching the curves.
    # (13, 10) is too vertically dominant and makes the legend harder to place.
    # Enkelte scenarier (E) har egen figsize i sin config-dict.
    "figsize"             : (16, 9),
    "show_overlay"        : True,
    "show_uscs"           : True,
    "show_sieve_openings" : True,
    "show_mgs1"           : False,
    "show_envelopes"      : False,
    "mgs1_auto_expand"    : True,
    "fines_end"           : 0.075,
    "sand_end"            : 4.75,
    "legend_loc"          : "upper left",
    "grid_alpha"          : 0.40,
    "reference_hydros"    : ("50A", "50B"),
    "reference_sieves"    : ("100W1", "100W2", "50W1", "50W2"),
}

# ── PSD-specific styling (built from mass-centric scheme) ────────────────────
# Each key maps to a style dict that plot_psd understands.
# Method inferred from key: 'W' → wet, 'F' prefix → fines-only hydro, else dry/bulk-hydro.

def make_psd_style(key, kind_override=None):
    """Build a style dict for a single PSD curve based on its test ID.
    kind_override: 'dry' | 'wet' | 'hydro_bulk' | 'hydro_fines' | None (auto)
    """
    mass = parse_mass(key)
    rep  = parse_rep_idx(key)
    s = str(key)
    if kind_override:
        method = kind_override
    elif s.startswith("F"):
        method = "hydro_fines"
    elif "W" in s:
        method = "wet"
    else:
        method = "dry"   # used for dry sieve and bulk hydro; call context distinguishes them
    return style_for(mass, method=method, replicate_idx=rep)

LINE_KW = {"linewidth": 2.0, "markersize": 6}
MGS1_MM = {"D10": 0.00519, "D30": 0.01996, "D50": 0.04930,
            "D60": 0.06685, "D90": 0.20548}

ENVELOPE_LABELS = {
    "A": "(A) Phoenix fines\n(Delage et al. (2022); Goetz et al. (2010))",
    "B": "(B) Active sands (Gale/Bagnold)\n(Golombek et al. (2020); Weitz et al. (2018))",
    "C": "(C) Armored/coarse lag\n(Blake et al. (2013); Weitz et al. (2018))",
}

# STYLE dict brukes av plot_psd (bakoverkompatibelt med gamle kall).
# Bygges automatisk fra make_psd_style() for alle test-ID-er.
STYLE = {}
for _key in ["50A","50B","10A","10B","2A","2B","0.5A","0.5B"]:
    STYLE[_key] = make_psd_style(_key, kind_override="dry")
for _key in ["100W1","100W2","50W1","50W2","10W1","10W2","10W3",
             "2W1","2W2","2W3","0.5W1","0.5W2","0.5W3"]:
    STYLE[_key] = make_psd_style(_key, kind_override="wet")
for _key in ["F0.5","F2","F10","F50"]:
    STYLE[_key] = make_psd_style(_key, kind_override="hydro_fines")


In [ ]:
# ── Overlay and annotation functions ────────────────────────────────────────

def add_uscs_header(ax, xlim, yband=(1.05, 1.10)):
    """USCS classification header above the PSD axes."""
    fines_end = PLOT_CFG["fines_end"]
    sand_end  = PLOT_CFG["sand_end"]
    lo, hi    = min(xlim), max(xlim)
    trans     = ax.get_xaxis_transform()
    y0, y1    = yband

    ax.add_patch(Rectangle((lo, y0), hi - lo, y1 - y0, transform=trans,
                            facecolor="white", edgecolor="0.75", lw=1.0,
                            clip_on=False, zorder=8))
    for label, xl, xr in [("FINES (silt + clay)", lo, fines_end),
                           ("SAND",                fines_end, sand_end),
                           ("GRAVEL",              sand_end, hi)]:
        xl_c = max(xl, lo); xr_c = min(xr, hi)
        if xr_c <= xl_c: continue
        ax.add_patch(Rectangle((xl_c, y0), xr_c - xl_c, y1 - y0, transform=trans,
                                facecolor="#f6f6f6", edgecolor="0.35", lw=1.0,
                                clip_on=False, zorder=9))
        xm = 10 ** ((np.log10(xl_c) + np.log10(xr_c)) / 2)
        ax.text(xm, (y0 + y1) / 2, label, ha="center", va="center",
                fontsize=11, transform=trans, zorder=10, clip_on=False)
    for xv in [fines_end, sand_end]:
        if lo < xv < hi:
            ax.axvline(xv, color="0.35", lw=1.0, zorder=0)

    # "USCS:" label just above the band — normal black text
    ax.text(0.002, y1 + 0.01, "USCS:",
            ha="left", va="bottom", fontsize=10,
            transform=ax.transAxes, zorder=11, clip_on=False)

def add_sieve_ticks(ax, sieve_mm, label_prefix="Sieve openings:"):
    """Sieve-opening tick marks on a secondary top axis."""
    ticks  = [float(t) for t in sieve_mm if t > 0]
    labels = [f"{t:g} mm" if t >= 1 else f"{int(round(t*1000))} µm" for t in ticks]
    ax_top = ax.twiny()
    ax_top.set_xscale("log")
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(ticks)
    ax_top.set_xticklabels(labels, fontsize=10)
    ax_top.tick_params(axis="x", which="both", length=4, pad=3)
    for sp in ax_top.spines.values(): sp.set_visible(False)
    ax_top.text(0.0, 1.02, label_prefix,
                transform=ax_top.transAxes, fontsize=10, va="bottom")
    return ax_top

def add_mgs1(ax, show=True,
             label="MGS-1 percentiles (Long-Fox & Britt (2023))"):
    """MGS-1 percentile markers connected by a line with D-value labels."""
    if not show: return
    xs     = np.array(list(MGS1_MM.values()))
    ys     = np.array([int(k[1:]) for k in MGS1_MM])
    xmin, xmax = ax.get_xlim()
    lo, hi = min(xmin, xmax), max(xmin, xmax)
    if PLOT_CFG["mgs1_auto_expand"]:
        lo = min(lo, xs.min() * 0.85); hi = max(hi, xs.max() * 1.15)
        ax.set_xlim(lo, hi)
    vis = (xs >= lo) & (xs <= hi)
    ax.semilogx(xs[vis], ys[vis], color="black", lw=1.5,
                marker="o", ms=7, markerfacecolor="black",
                label=label, zorder=11)
    for d_key, d_val in MGS1_MM.items():
        y_val = int(d_key[1:])
        if lo <= d_val <= hi:
            ax.annotate(d_key, xy=(d_val, y_val),
                        xytext=(-5, 6), textcoords="offset points",
                        ha="right", fontsize=10, fontweight="bold", zorder=12)

def add_mars_envelopes(ax, show=True, full_labels=False):
    """Mars soil particle-size envelopes as background bands."""
    if not show: return
    x       = np.logspace(np.log10(0.001), np.log10(6.0), 600)
    anchors = np.array([0.002, 0.010, 0.050, 0.150, 0.500, 2.000, 5.000])
    env_data = {
        "A": (np.array([2,10,40,80,98,100,100]), np.array([5,20,60,95,100,100,100])),
        "B": (np.array([0,2,35,65,90,99,100]),   np.array([1,5,45,80,97,100,100])),
        "C": (np.array([0,0,10,30,60,90,97]),    np.array([0,1,20,45,80,98,99])),
    }
    colors = {"A": "tab:blue", "B": "tab:orange", "C": "tab:green"}
    for key, (y_lo, y_hi) in env_data.items():
        ylo = np.interp(np.log10(x), np.log10(anchors), y_lo)
        yhi = np.interp(np.log10(x), np.log10(anchors), y_hi)
        lbl = ENVELOPE_LABELS[key] if full_labels else f"({key})"
        ax.fill_between(x, ylo, yhi, alpha=0.22, color=colors[key],
                        label=lbl, zorder=1)

def setup_psd_ax(ax, xlim=None, ylim=None,
                 xlabel="Particle diameter (mm)",
                 ylabel="Percent finer [%]"):
    """Configure log x-scale axis with standard PSD labels."""
    xlim = PLOT_CFG["xlim_full"] if xlim is None else xlim
    ylim = PLOT_CFG["ylim"]      if ylim  is None else ylim
    ax.set_xscale("log")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, which="both", alpha=PLOT_CFG["grid_alpha"])

def apply_psd_overlay(ax, xlim=None, sieve_mm=None, sieve_label="Sieve openings:"):
    """Add USCS header and sieve-opening ticks.
    Pass sieve_mm to override the default experimental sieves (e.g. ASTM sieves).
    """
    xlim    = ax.get_xlim() if xlim is None else xlim
    _sieves = SIEVES_MM[SIEVES_MM > 0] if sieve_mm is None else np.asarray(sieve_mm)
    if PLOT_CFG.get("show_sieve_openings", True):
        ticks_in = [t for t in _sieves if min(xlim) <= t <= max(xlim)]
        if ticks_in: add_sieve_ticks(ax, ticks_in, label_prefix=sieve_label)
    if PLOT_CFG.get("show_uscs", True):
        add_uscs_header(ax, xlim)


### 3.1 Sieve and Hydrometer Input Data


In [ ]:
# Figure/Table input: dry and wet-prepared sieve retained masses.
SIEVES_MM = np.array([2.0, 1.0, 0.5, 0.25, 0.125, 0.075, 0.0])

def load_sieve_retained(filename):
    raw = pd.read_csv(DATA_DIR / filename)
    out = {}
    for test_id, sub in raw.groupby("test_id", sort=False):
        sub = sub.sort_values("sieve_order")
        out[test_id] = sub["retained_g"].to_numpy(float)
    return out

PSD_DATA = {}
PSD_DATA.update(load_sieve_retained("dry_sieve_raw.csv"))
PSD_DATA.update(load_sieve_retained("wet_sieve_raw.csv"))

# Figure/Table input: bulk hydrometer readings.
hydro_raw = pd.read_csv(DATA_DIR / "hydrometer_bulk_raw.csv")
PSD_HYDRO_RUNS = {
    test_id: {
        "wet_mass_g": float(sub["wet_mass_g"].iloc[0]),
        "data": list(sub[["t_min", "H", "T_C"]].itertuples(index=False, name=None)),
    }
    for test_id, sub in hydro_raw.groupby("test_id", sort=False)
}

hydro_params = pd.read_csv(DATA_DIR / "hydrometer_constants.csv").set_index("parameter")["value"]
HYD_RHO_S = float(hydro_params["rho_s_hydro"])
HYD_WC_PCT = float(hydro_params["water_content_hydro_pct"])


### 3.2 Hydrometer Calculations


In [ ]:
def load_hydro_constants(path):
    values = pd.read_csv(path).set_index("parameter")["value"].to_dict()
    return {k: float(v) for k, v in values.items()}

def wet_to_dry(wet_g, wc_pct):
    return wet_g * 100.0 / (100.0 + wc_pct)

def Lcorr(T_C, c):
    return c["B_152"] - c["a1"] * T_C - c["a2"] * T_C**2

def Leff(H, c):
    return (c["R36"]
            + (c["R35"] - c["R36"]) / (c["U36"] - c["U35"]) * (c["U36"] - H + c["O37"])
            - c["O35"] / (2 * c["O36"]))

def D_mm(t_min, H, Gs_T, c):
    if t_min <= 0:
        return np.nan
    denom = c["rho_w"] * c["g"] * (Gs_T - 1.0) * t_min * 60.0
    return np.sqrt(18.0 * c["mu"] * Leff(H, c) / denom) * 10.0 if denom > 0 else np.nan

def pct_finer_raw(t_min, H, T_C, Md_g, Gs_T, c):
    if Md_g <= 0:
        return np.nan
    Lc = Lcorr(T_C, c)
    return c["coef_pct"] * (Gs_T / (Gs_T - 1.0)) * (1000.0 / Md_g) * (H - Lc)

def compute_hydro_run(name, run_data, wc_pct, rho_s, constants):
    Md = wet_to_dry(run_data["wet_mass_g"], wc_pct)
    Gs_T = rho_s / constants["rho_w"]
    rows = []
    for (t, H, T) in run_data["data"]:
        rows.append({
            "test": name, "t_min": t, "H": H, "T_C": T, "Md_g": Md, "Gs_T": Gs_T,
            "Lcorr": Lcorr(T, constants),
            "D_mm": D_mm(t, H, Gs_T, constants),
            "%finer_raw": pct_finer_raw(t, H, T, Md, Gs_T, constants),
        })
    df = pd.DataFrame(rows).sort_values("D_mm", ascending=False).reset_index(drop=True)
    df["%finer"] = df["%finer_raw"].clip(0, 100)
    df["over_100"] = df["%finer_raw"] > 100
    df["signal"] = df["H"] - df["Lcorr"]
    df["low_signal"] = df["signal"] < 0.5
    return df

HYDRO_CONST = load_hydro_constants(DATA_DIR / "hydrometer_astm152h_constants.csv")
HYDRO_WITH = {
    k: compute_hydro_run(k, v, HYD_WC_PCT, HYD_RHO_S, HYDRO_CONST)
    for k, v in PSD_HYDRO_RUNS.items()
}
print("Bulk hydrometer runs:", list(HYDRO_WITH.keys()))


### 3.3 Curve-Building and Gradation Metrics


In [ ]:
# ── Sieve curve from retained mass ───────────────────────────────────────────────
def sieve_curve(key):
    """Return (d_mm, pct_finer, total_g) for a sieve test."""
    ret   = np.asarray(PSD_DATA[key], float)
    total = ret.sum()
    cum   = np.cumsum(ret)
    mask  = SIEVES_MM > 0
    x = SIEVES_MM[mask]
    y = 100 * (1 - cum[mask] / total)
    return x, y, total

def p075(key):
    """Fraction finer than 0.075 mm for a sieve test."""
    _, y, _ = sieve_curve(key)
    return float(y[-1])

def hydro_curve(key, src=None):
    """Return (d_mm, pct_finer) for a hydrometer test."""
    src = HYDRO_WITH if src is None else src
    df  = src[key]
    d   = pd.to_numeric(df["D_mm"], errors="coerce").to_numpy(float)
    p   = pd.to_numeric(df["%finer"], errors="coerce").to_numpy(float)
    m   = np.isfinite(d) & np.isfinite(p) & (d > 0)
    d, p = d[m], p[m]
    idx  = np.argsort(d)
    return d[idx], np.clip(p[idx], 0, 100)

def D_at_pct(d_mm, p_finer, target):
    """Interpolate particle diameter at a given percent finer."""
    d, p = np.asarray(d_mm, float), np.asarray(p_finer, float)
    m    = np.isfinite(d) & np.isfinite(p) & (d > 0)
    d, p = d[m], p[m]
    if len(d) < 2: return np.nan
    idx  = np.argsort(d)
    d, p = d[idx], p[idx]
    if p[0] > p[-1]: p = 100.0 - p
    if target < p.min() or target > p.max(): return np.nan
    return 10 ** np.interp(target, p, np.log10(d))

def gradation_metrics(d_mm, p_finer):
    """Beregner D10, D30, D50, D60, D90, Cu og Cc."""
    D = {f"D{n}": D_at_pct(d_mm, p_finer, n) for n in [10, 30, 50, 60, 90]}
    D10, D30, D60 = D["D10"], D["D30"], D["D60"]
    Cu = D60 / D10 if all(np.isfinite([D10, D60])) and D10 > 0 else np.nan
    Cc = D30**2 / (D10 * D60) if all(np.isfinite([D10, D30, D60])) and D10*D60 > 0 else np.nan
    return {**D, "Cu": Cu, "Cc": Cc}

def stitch_psd(hydro_key, sieve_key, P075_val=None):
    """Combine hydrometer and sieve curves into a full PSD."""
    if P075_val is None: P075_val = p075(sieve_key)
    x_s, y_s, _ = sieve_curve(sieve_key)
    y_s          = y_s.copy(); y_s[-1] = P075_val
    d_h, p_h     = hydro_curve(hydro_key)
    keep         = d_h < PLOT_CFG["fines_end"]
    d_h, p_h     = d_h[keep], p_h[keep]
    d_all = np.concatenate([d_h, [PLOT_CFG["fines_end"]], x_s])
    p_all = np.concatenate([p_h, [P075_val], y_s])
    df    = pd.DataFrame({"d_mm": d_all, "p": p_all}).dropna()
    df    = df.sort_values("d_mm").drop_duplicates("d_mm", keep="first")
    return df["d_mm"].to_numpy(float), df["p"].to_numpy(float)

def interp_to_grid(d_mm, p, grid):
    d, y = np.asarray(d_mm, float), np.asarray(p, float)
    m    = np.isfinite(d) & np.isfinite(y) & (d > 0)
    d, y = d[m], y[m]
    if len(d) < 2: return np.full_like(grid, np.nan, float)
    idx  = np.argsort(d)
    return np.interp(np.log10(grid), np.log10(d[idx]), np.clip(y[idx], 0, 100),
                     left=np.nan, right=np.nan)


### 3.4 Plotting Helpers


In [ ]:
# ── Generic PSD plotting function ────────────────────────────────────────────
def plot_psd(curves, xlim=None, ylim=None,
             show_overlay=None, show_mgs1=None, show_envelopes=None):
    """
    Plot one or more PSD curves.

    Parameters
    ----------
    curves : list of dicts with keys 'label', 'd_mm', 'p', and optional 'style'.
    """
    cfg            = PLOT_CFG
    show_overlay   = cfg["show_overlay"]   if show_overlay   is None else show_overlay
    show_mgs1      = cfg["show_mgs1"]      if show_mgs1      is None else show_mgs1
    show_envelopes = cfg["show_envelopes"] if show_envelopes is None else show_envelopes

    # Use PSD-specific figsize (wider, to give USCS zones room)
    fig, ax = plt.subplots(figsize=cfg.get("figsize", plt.rcParams["figure.figsize"]))
    add_mars_envelopes(ax, show=show_envelopes, full_labels=show_envelopes)

    for c in curves:
        st = {**LINE_KW, **STYLE.get(c["label"], {}), **c.get("style", {})}
        ax.semilogx(c["d_mm"], c["p"], label=c["label"], **st)

    setup_psd_ax(ax, xlim=xlim, ylim=ylim)
    add_mgs1(ax, show=show_mgs1)
    if show_overlay: apply_psd_overlay(ax)

    ax.legend(frameon=False, loc=cfg["legend_loc"])
    plt.tight_layout()
    return fig, ax

def plot_sieve(keys, xlim=None, ylim=None,
               show_overlay=None, show_mgs1=None, show_envelopes=None):
    """Plot sieve curves for the given test IDs."""
    curves = [{"label": k, "d_mm": sieve_curve(k)[0], "p": sieve_curve(k)[1]} for k in keys]
    return plot_psd(curves, xlim=xlim or PLOT_CFG["xlim_sieve"],
                    ylim=ylim, show_overlay=show_overlay,
                    show_mgs1=show_mgs1, show_envelopes=show_envelopes)

def plot_hydrometer(keys, xlim=None, ylim=None,
                    show_overlay=None, show_mgs1=None, show_envelopes=None):
    """Plot hydrometer curves for the given test IDs."""
    curves = [{"label": k, "d_mm": hydro_curve(k)[0], "p": hydro_curve(k)[1]} for k in keys]
    return plot_psd(curves, xlim=xlim or PLOT_CFG["xlim_hydro"],
                    ylim=ylim, show_overlay=show_overlay,
                    show_mgs1=show_mgs1, show_envelopes=show_envelopes)

def plot_stitched(combos, xlim=None, ylim=None,
                  show_overlay=None, show_mgs1=None, show_envelopes=None):
    """Plot combined (stitched) PSD curves.
    combos: list of (hydro_key, sieve_key, P075, label)
    """
    curves = []
    for hk, sk, P075, lbl in combos:
        d, p = stitch_psd(hk, sk, P075)
        curves.append({"label": lbl, "d_mm": d, "p": p})
    return plot_psd(curves, xlim=xlim or PLOT_CFG["xlim_full"],
                    ylim=ylim, show_overlay=show_overlay,
                    show_mgs1=show_mgs1, show_envelopes=show_envelopes)

def plot_sieve_group_medians(groups, xlim=None, ylim=None, band=None,
                              show_overlay=None, show_mgs1=None, show_envelopes=None):
    """Plot median PSD per group with optional min-max band.
    groups: dict {'label': [keys]}
    band  : None, 'minmax', or 'iqr'
    """
    cfg            = PLOT_CFG
    show_overlay   = cfg["show_overlay"]   if show_overlay   is None else show_overlay
    show_mgs1      = cfg["show_mgs1"]      if show_mgs1      is None else show_mgs1
    show_envelopes = cfg["show_envelopes"] if show_envelopes is None else show_envelopes

    fig, ax = plt.subplots(figsize=cfg.get("figsize", plt.rcParams["figure.figsize"]))
    add_mars_envelopes(ax, show=show_envelopes, full_labels=show_envelopes)

    for grp_label, keys in groups.items():
        x_ref, Y = None, []
        for k in keys:
            x, y, _ = sieve_curve(k)
            if x_ref is None: x_ref = x
            Y.append(y)
        Y     = np.vstack(Y)
        y_med = np.nanmedian(Y, axis=0)
        base_st = {**LINE_KW, **STYLE.get(keys[0], {})}
        color   = base_st.get("color", None)
        ax.plot(x_ref, y_med, label=grp_label, **base_st)
        if band == "minmax":
            ax.fill_between(x_ref, Y.min(0), Y.max(0), alpha=0.12, color=color)
        elif band == "iqr":
            ax.fill_between(x_ref, np.nanpercentile(Y,25,0),
                            np.nanpercentile(Y,75,0), alpha=0.12, color=color)

    setup_psd_ax(ax, xlim=xlim or PLOT_CFG["xlim_sieve"], ylim=ylim)
    add_mgs1(ax, show=show_mgs1)
    if show_overlay: apply_psd_overlay(ax)

    ax.legend(frameon=False, loc=cfg["legend_loc"])
    plt.tight_layout()
    return fig, ax

def build_reference_ensemble(hydro_keys=None, sieve_keys=None):
    """Build all reference stitched PSD scenarios."""
    h_keys = PLOT_CFG["reference_hydros"] if hydro_keys is None else hydro_keys
    s_keys = PLOT_CFG["reference_sieves"] if sieve_keys is None else sieve_keys
    scenarios = []
    for h in h_keys:
        for s in s_keys:
            P075_val = p075(s)
            d, p_    = stitch_psd(h, s, P075_val)
            scenarios.append({"hydro": h, "sieve": s, "P075": P075_val, "d_mm": d, "p": p_})
    return scenarios

def plot_reference_band(scenarios, xlim=None, ylim=None, show_median=True,
                        show_overlay=None, show_mgs1=None, show_envelopes=None):
    """Plot min-max band from reference scenarios with optional median."""
    cfg = PLOT_CFG
    xlim = xlim or cfg["xlim_full"]; ylim = ylim or cfg["ylim"]
    show_overlay   = cfg["show_overlay"]   if show_overlay   is None else show_overlay
    show_mgs1      = cfg["show_mgs1"]      if show_mgs1      is None else show_mgs1
    show_envelopes = cfg["show_envelopes"] if show_envelopes is None else show_envelopes

    grid = np.logspace(np.log10(xlim[0]), np.log10(xlim[1]), 500)
    Y    = np.vstack([interp_to_grid(sc["d_mm"], sc["p"], grid) for sc in scenarios])

    fig, ax = plt.subplots(figsize=cfg.get("figsize", plt.rcParams["figure.figsize"]))
    add_mars_envelopes(ax, show=show_envelopes, full_labels=show_envelopes)
    ax.fill_between(grid, np.nanmin(Y,0), np.nanmax(Y,0), alpha=0.30, color="tab:gray", label="Min–max band")
    if show_median:
        ax.semilogx(grid, np.nanmedian(Y,0), lw=2, label="Median")

    setup_psd_ax(ax, xlim=xlim, ylim=ylim)
    add_mgs1(ax, show=show_mgs1)
    if show_overlay: apply_psd_overlay(ax)

    ax.legend(frameon=False, loc=cfg["legend_loc"])
    plt.tight_layout()
    return fig, ax


### 3.5 Dry Sieving


In [ ]:
# ── Figure: Small dry sieve specimins ─────────────────────────────────────
SMALL_DRY = ["2A", "2B", "0.5A", "0.5B"]
SIEVE_XLIM = PLOT_CFG["xlim_sieve"]
fig, ax = plot_sieve(SMALL_DRY, xlim=SIEVE_XLIM)
savefig(fig, "fig_3_5_dry_sieve_2_05")
plt.show()


In [ ]:
# ── Figure: Big dry sieve specimins ─────────────────────────────────────────
BIG_DRY = ["50A", "50B", "10A", "10B"]
SIEVE_XLIM = PLOT_CFG["xlim_sieve"]
fig, ax = plot_sieve(BIG_DRY, xlim=SIEVE_XLIM)
savefig(fig, "fig_3_4_dry_sieve_50_10")
plt.show()


In [ ]:
# ── Figure: Median per mass group (dry sieve) ────────────────────────────────
dry_groups = {
    "50 g dry" : ["50A", "50B"],
    "10 g dry" : ["10A", "10B"],
    "2 g dry"  : ["2A",  "2B"],
    "0.5 g dry": ["0.5A","0.5B"],
}

fig, ax = plot_sieve_group_medians(dry_groups, xlim=SIEVE_XLIM)
savefig(fig, "fig_3_6_dry_sieve_medians")
plt.show()


### 3.6 Wet-Prepared Sieving


In [ ]:
# ── Figures: All wet-sieve specimens ────────────────────────────────────────
WET_LARGE = ["100W1", "100W2", "50W1", "50W2", "10W1", "10W2", "10W3"]
WET_SMALL = ["2W1",   "2W2",   "2W3",  "0.5W1","0.5W2","0.5W3"]
WET_ALL   = WET_LARGE + WET_SMALL

fig, ax = plot_sieve(WET_LARGE, xlim=SIEVE_XLIM)
savefig(fig, "fig_3_7_wet_sieve_100_50_10")
plt.show()

fig, ax = plot_sieve(WET_SMALL, xlim=SIEVE_XLIM)
savefig(fig, "fig_3_8_wet_sieve_2_05")
plt.show()


In [ ]:
# ── Figure: Median per mass group (wet sieve) ────────────────────────────────
wet_groups = {
    "100 g wet": ["100W1","100W2"],
    "50 g wet" : ["50W1", "50W2"],
    "10 g wet" : ["10W1", "10W2","10W3"],
    "2 g wet"  : ["2W1",  "2W2", "2W3"],
    "0.5 g wet": ["0.5W1","0.5W2","0.5W3"],
}

fig, ax = plot_sieve_group_medians(wet_groups, band= None, xlim=SIEVE_XLIM)
savefig(fig, "fig_3_9_wet_sieve_medians")
plt.show()


### 3.7 Bulk Hydrometer


In [ ]:
# ── Figure: All hydrometer specimens ────────────────────────────────────────
HYDRO_KEYS = ["50A", "50B", "10A", "10B", "2A", "2B", "0.5A", "0.5B"]
HYDRO_XLIM = PLOT_CFG["xlim_hydro"]

fig, ax = plot_hydrometer(HYDRO_KEYS, xlim=HYDRO_XLIM)
ax.set_xlim(0.0009, 0.075)
savefig(fig, "fig_3_13_hydro_bulk")
plt.show()


In [ ]:
# ── Hydrometer QC table ──────────────────────────────────────────────────────
ref_hydro = ["50A", "50B"]
grid_qc   = np.logspace(np.log10(0.003), np.log10(0.05), 60)

ref_y = np.nanmean(np.vstack([
    interp_to_grid(*hydro_curve(k), grid_qc) for k in ref_hydro
]), axis=0)

qc_rows = []
for k in HYDRO_KEYS:
    df_h = HYDRO_WITH[k]
    y_k  = interp_to_grid(*hydro_curve(k), grid_qc)
    qc_rows.append({
        "Test ID"              : k,
        "Dry mass [g]"         : round(float(df_h["Md_g"].iloc[0]), 4),
        "Max (H – Lcorr)"      : round(float(df_h["signal"].max()), 2),
        "Low signal [%]"       : round(float(df_h["low_signal"].mean()) * 100, 1),
        "RMS vs 50g ref [pp]"  : round(float(np.sqrt(np.nanmean((y_k - ref_y)**2))), 2),
    })

print("Hydrometer QC table:")
display(pd.DataFrame(qc_rows).set_index("Test ID"))


### 3.8 Fines-Only Hydrometer Check


In [ ]:
# Figure/Table input: fines-only hydrometer readings.
fines_raw = pd.read_csv(DATA_DIR / "hydrometer_fines_raw.csv")
FINES_HYDRO_RUNS = {
    test_id: {
        "fines_mass_g": float(sub["fines_mass_g"].iloc[0]),
        "bulk_nominal_g": float(sub["bulk_nominal_g"].iloc[0]),
        "data": list(sub[["t_min", "H", "T_C"]].itertuples(index=False, name=None)),
    }
    for test_id, sub in fines_raw.groupby("test_id", sort=False)
}

STYLE.update({
    "F0.5": {"color": "tab:red",    "marker": "^", "linestyle": "-", "linewidth": 1.2},
    "F2":   {"color": "tab:green",  "marker": "^", "linestyle": "-", "linewidth": 1.2},
    "F10":  {"color": "tab:orange", "marker": "^", "linestyle": "-", "linewidth": 1.2},
    "F50":  {"color": "tab:blue",   "marker": "^", "linestyle": "-", "linewidth": 1.2},
})

HYDRO_FINES = {}
for key, run in FINES_HYDRO_RUNS.items():
    fake = {"wet_mass_g": run["fines_mass_g"], "data": run["data"]}
    HYDRO_FINES[key] = compute_hydro_run(
        key, fake, wc_pct=0.0, rho_s=HYD_RHO_S, constants=HYDRO_CONST
    )

P075_SCALE = np.median([p075(k) for k in ["100W1", "100W2", "50W1", "50W2"]]) / 100.0

for key, df in HYDRO_FINES.items():
    p_raw = (
        HYDRO_CONST["coef_pct"]
        * (df["Gs_T"] / (df["Gs_T"] - 1.0))
        * (1000.0 / df["Md_g"])
        * (df["H"] - df["Lcorr"])
    )
    df["%finer_raw_rebuilt"] = p_raw
    df["%finer_bulk_raw"] = p_raw * P075_SCALE
    df["%finer_bulk"] = np.clip(df["%finer_bulk_raw"], 0, 100)
    df["over_100_bulk"] = df["%finer_bulk_raw"] > 100

print(f"P(0.075 mm) scaling factor: {P075_SCALE:.3f} ({P075_SCALE*100:.1f} %)")
print("Fines-only runs computed:", list(HYDRO_FINES.keys()))

rows = []
for key, run in FINES_HYDRO_RUNS.items():
    df = HYDRO_FINES[key]
    rows.append({
        "ID": key,
        "Fines mass [g]": run["fines_mass_g"],
        "Bulk nominal [g]": run["bulk_nominal_g"],
        "Bulk equiv. [g]": round(run["fines_mass_g"] / P075_SCALE, 2),
        "Max signal (H-Lc)": round(float(df["signal"].max()), 2),
        "Low signal [%]": round(float(df["low_signal"].mean()) * 100, 1),
        "Max raw fines [%]": round(float(df["%finer_raw_rebuilt"].max()), 1),
        "Max bulk-scaled [%]": round(float(df["%finer_bulk_raw"].max()), 1),
        "Pts >100 bulk-scaled": int(df["over_100_bulk"].sum()),
    })
display(pd.DataFrame(rows).set_index("ID"))

# Figure 3.14 - bulk hydrometer and fines-only hydrometer, bulk-equivalent basis.
HYDRO_XLIM = (0.001, 0.2)
bulk_keys = ["50A", "50B", "10A", "10B", "2A", "2B", "0.5A", "0.5B"]
curves = []

for k in bulk_keys:
    d, p = hydro_curve(k)
    curves.append({
        "label": f"{k} (bulk)",
        "d_mm": d,
        "p": p,
        "style": {**STYLE.get(k, {}), "linestyle": "--", "alpha": 0.55},
    })

for k in FINES_HYDRO_RUNS:
    df = HYDRO_FINES[k].sort_values("D_mm")
    label = f"{FINES_HYDRO_RUNS[k]['fines_mass_g']:g} g fines (≈{FINES_HYDRO_RUNS[k]['bulk_nominal_g']:g} g bulk)"
    curves.append({
        "label": label,
        "d_mm": df["D_mm"].to_numpy(float),
        "p": df["%finer_bulk"].to_numpy(float),
        "style": {**STYLE.get(k, {})},
    })

fig, ax = plot_psd(
    curves,
    xlim=HYDRO_XLIM,
    show_overlay=False,
    show_mgs1=False,
    show_envelopes=False,
)
add_uscs_header(ax, HYDRO_XLIM)
ax.set_xlim(*HYDRO_XLIM)
ax.legend(frameon=False, loc="upper right", fontsize="small")
savefig(fig, "fig_3_14_hydro_bulk_vs_fines")
plt.show()


In [ ]:
# ── Figure: Signal strength (H – Lcorr) over time – bulk vs. fines-only ──────
# Higher values = stronger hydrometer signal = more reliable reading.
# Shows whether fines-only concentration gives better signal for small masses.

fig, axes = plt.subplots(2, 2,
                          figsize=(PLOT_CFG["figsize"][0],
                                   PLOT_CFG["figsize"][1] * 1.3),
                          sharex=True, sharey=True)
axes = axes.flatten()

pair_map = [
    ("F0.5", ["0.5A","0.5B"]),
    ("F2",   ["2A",  "2B"]),
    ("F10",  ["10A", "10B"]),
    ("F50",  ["50A", "50B"]),
]

for ax, (fkey, bkeys) in zip(axes, pair_map):
    nom = FINES_HYDRO_RUNS[fkey]["bulk_nominal_g"]

    # Bulk specimens
    for bk in bkeys:
        df = HYDRO_WITH[bk]
        ax.semilogx(df["t_min"], df["signal"], **{**STYLE.get(bk,{}),
                    "label": f"{bk} (bulk)", "linewidth": 1.8})

    # Fines-only specimen
    df_f = HYDRO_FINES[fkey]
    fm   = FINES_HYDRO_RUNS[fkey]["fines_mass_g"]
    ax.semilogx(df_f["t_min"], df_f["signal"], **{**STYLE.get(fkey,{}),
                "label": f"{fkey} ({fm:g} g fines)", "linewidth": 1.0})

    ax.axhline(0.5, color="gray", ls=":", lw=1, label="Min reliable (0.5)")
    ax.set_title(f"≈ {nom:g} g bulk", fontsize=14)
    ax.grid(True, which="both", alpha=0.35)
    ax.legend(fontsize=14, frameon=False)

for ax in axes:
    ax.set_xlabel("Time [min]")
axes[0].set_ylabel("Signal, H – Lcorr")
axes[2].set_ylabel("Signal, H – Lcorr")
fig.suptitle("Hydrometer signal strength: bulk vs. fines-only", y=1.01)
plt.tight_layout()
savefig(fig, "fig_3_15_hydro_signal")
plt.show()


### 3.9 Stitched Full PSD


Median bulk-hydrometer curves are stitched to median wet-prepared sieve curves at 0.075 mm.


In [ ]:
FULL_XLIM = PLOT_CFG["xlim_full"]

# Scenario E - two thesis figures, B&W readable.
#
# Figure 3.16: median hydrometer + wet sieve for 50, 10, 2, 0.5 g
#              without MGS-1 or Mars envelopes.
# Figure 3.17: median hydrometer + wet sieve for 50 and 10 g
#              with MGS-1 line and Mars envelopes.

LINESTYLE_BY_MASS = {
    50:  "-",
    10:  (0, (6, 2)),
    2:   "-.",
    0.5: (0, (1, 1.5)),
}
MGS1_LINESTYLE = (0, (5, 2, 1, 2))
MGS1_LW = 2.4
MASS_LW = 1.8

HYDRO_GROUPS = {
    50 : ["50A",  "50B"],
    10 : ["10A",  "10B"],
    2  : ["2A",   "2B"],
    0.5: ["0.5A", "0.5B"],
}
WET_GROUPS = {
    100: ["100W1", "100W2"],
    50 : ["50W1",  "50W2"],
    10 : ["10W1",  "10W2", "10W3"],
    2  : ["2W1",   "2W2",  "2W3"],
    0.5: ["0.5W1", "0.5W2","0.5W3"],
}

def _median_sieve_curve(keys):
    """Median wet-sieve curve over replicate keys. Returns (d_mm, p_finer, p075_median)."""
    Ys, x_ref = [], None
    for k in keys:
        x, y, _ = sieve_curve(k)
        Ys.append(y)
        x_ref = x if x_ref is None else x_ref
    Y = np.vstack(Ys)
    return x_ref, np.nanmedian(Y, axis=0), float(np.nanmedian(Y[:, -1]))

def _median_bulk_hydro(keys):
    """Median bulk-hydrometer curve across replicate keys on a common diameter grid."""
    grid = np.logspace(np.log10(0.001), np.log10(0.075), 120)
    Ys = [interp_to_grid(*hydro_curve(k), grid) for k in keys]
    return grid, np.nanmedian(np.vstack(Ys), axis=0)

def _stitch_arrays(d_h, p_h, d_s, p_s, P075_val):
    """Stitch hydrometer and sieve arrays at 0.075 mm."""
    keep = d_h < PLOT_CFG["fines_end"]
    d_h2, p_h2 = d_h[keep], p_h[keep]
    y_s2 = p_s.copy()
    y_s2[-1] = P075_val
    d_all = np.concatenate([d_h2, [PLOT_CFG["fines_end"]], d_s])
    p_all = np.concatenate([p_h2, [P075_val], y_s2])
    df = pd.DataFrame({"d_mm": d_all, "p": p_all}).dropna()
    df = df.sort_values("d_mm").drop_duplicates("d_mm", keep="first")
    return df["d_mm"].to_numpy(float), df["p"].to_numpy(float)

def add_mgs1_line_only(ax,
                       d_um=(5.19, 19.96, 49.30, 66.85, 205.48),
                       p_pct=(10, 30, 50, 60, 90),
                       color="black", lw=MGS1_LW, ls=MGS1_LINESTYLE,
                       label="MGS-1 (Long-Fox & Britt, 2023)"):
    """Plot MGS-1 reference percentiles as a line, without markers."""
    d_mm = np.asarray(d_um, dtype=float) / 1000.0
    ax.plot(d_mm, p_pct, color=color, lw=lw, ls=ls, label=label, zorder=4)

def build_curve_for_mass(m):
    """Build one median stitched curve for a nominal mass level."""
    hydro_keys = HYDRO_GROUPS.get(m, [])
    wet_keys = WET_GROUPS.get(m, [])
    if not hydro_keys or not wet_keys:
        print(f"  -> missing hydrometer or wet-sieve data at {m} g; skipped")
        return None
    d_h, p_h = _median_bulk_hydro(hydro_keys)
    d_s, p_s, P075_med = _median_sieve_curve(wet_keys)
    d_st, p_st = _stitch_arrays(d_h, p_h, d_s, p_s, P075_med)
    st = style_for(m, method="stitched", replicate_idx=0,
                   linestyle=LINESTYLE_BY_MASS.get(m, "--"), alpha=0.9)
    st.update(marker="None",
              linestyle=LINESTYLE_BY_MASS.get(m, "--"),
              linewidth=MASS_LW)
    label = f"{m} g bulk - median hydrometer + wet sieve"
    return {"label": label, "d_mm": d_st, "p": p_st, "style": st,
            "mass": m, "P075": P075_med}

# Figure 3.16 - all stitched mass levels, no MGS-1, no envelopes.
masses_fig1 = [50, 10, 2, 0.5]
curves_fig1 = [c for c in (build_curve_for_mass(m) for m in masses_fig1)
               if c is not None]

fig1, ax1 = plot_psd(curves_fig1, xlim=FULL_XLIM)
fig1.set_size_inches(16, 9)
apply_psd_overlay(ax1)
ax1.legend(frameon=False, loc="lower right", fontsize=13,
           bbox_to_anchor=(0.94, 0.0))
plt.tight_layout()
savefig(fig1, "fig_3_16_stitched_full")
plt.show()

# Figure 3.17 - 50 and 10 g stitched curves with MGS-1 line and Mars envelopes.
masses_fig2 = [50, 10]
curves_fig2 = [c for c in (build_curve_for_mass(m) for m in masses_fig2)
               if c is not None]

fig2, ax2 = plot_psd(curves_fig2, xlim=FULL_XLIM)
fig2.set_size_inches(16, 9)
apply_psd_overlay(ax2)
add_mgs1_line_only(ax2)
add_mars_envelopes(ax2, show=True, full_labels=True)
ax2.legend(frameon=False, loc="lower right", fontsize=13,
           bbox_to_anchor=(0.94, 0.0))
plt.tight_layout()
savefig(fig2, "fig_3_17_stitched_vs_reference")
plt.show()

PERCENTILES = (10, 30, 50, 60, 90)

def _d_at_passing(d_mm, p_pct, target):
    """Diameter at a given percent passing, interpolated in log10(d)."""
    d = np.asarray(d_mm, dtype=float)
    p = np.asarray(p_pct, dtype=float)
    mask = np.isfinite(d) & np.isfinite(p) & (d > 0)
    d, p = d[mask], p[mask]
    if d.size < 2:
        return np.nan
    order = np.argsort(p)
    p_s, d_s = p[order], d[order]
    if target < p_s.min() or target > p_s.max():
        return np.nan
    log_d = np.interp(target, p_s, np.log10(d_s))
    return float(10 ** log_d)

def gradation_table(curves):
    """Standard gradation metrics for the per-mass median stitched curves."""
    rows = []
    for sc in curves:
        d_mm, p_pct = sc["d_mm"], sc["p"]
        Ds = {f"D{q}_mm": _d_at_passing(d_mm, p_pct, q) for q in PERCENTILES}
        D10, D30, D60 = Ds["D10_mm"], Ds["D30_mm"], Ds["D60_mm"]
        Cu = D60 / D10 if (np.isfinite(D60) and np.isfinite(D10) and D10 > 0) else np.nan
        Cc = (D30**2) / (D10 * D60) if all(np.isfinite([D10, D30, D60])) and D10 > 0 and D60 > 0 else np.nan
        rows.append({
            "Mass [g]": sc["mass"],
            "P0.075 [%]": round(sc["P075"], 2),
            "D10 [mm]": round(Ds["D10_mm"], 4) if np.isfinite(Ds["D10_mm"]) else np.nan,
            "D30 [mm]": round(Ds["D30_mm"], 4) if np.isfinite(Ds["D30_mm"]) else np.nan,
            "D50 [mm]": round(Ds["D50_mm"], 4) if np.isfinite(Ds["D50_mm"]) else np.nan,
            "D60 [mm]": round(Ds["D60_mm"], 4) if np.isfinite(Ds["D60_mm"]) else np.nan,
            "D90 [mm]": round(Ds["D90_mm"], 4) if np.isfinite(Ds["D90_mm"]) else np.nan,
            "Cu": round(Cu, 3) if np.isfinite(Cu) else np.nan,
            "Cc": round(Cc, 3) if np.isfinite(Cc) else np.nan,
        })
    return pd.DataFrame(rows).sort_values("Mass [g]", ascending=False).reset_index(drop=True)

df_grad = gradation_table(curves_fig1)
print("Gradation metrics - per-mass median stitched curves:")
display(df_grad)
Path("tables").mkdir(exist_ok=True)
df_grad.to_csv("tables/table_3_7_gradation_stitched.csv", index=False)


Figure 3.17 is generated together with Figure 3.16 in the previous cell.


### 3.10 Sieve Metrics: P(0.075 mm) and D50


In [ ]:
# ── Helper: parse specimen mass from test ID ─────────────────────────────────
def parse_specimen_mass(key):
    """Extract nominal specimen mass [g] from a test ID string."""
    s = str(key).replace(",", ".")
    m = re.match(r"^(\d+(?:\.\d+)?)", s)
    return float(m.group(1)) if m else np.nan

def group_stat(keys, val_fn, stat="median"):
    """Compute per-mass statistics for a list of keys."""
    by_mass = {}
    for k in keys:
        v = val_fn(k)
        by_mass.setdefault(parse_specimen_mass(k), []).append(v)
    masses = sorted(by_mass)
    center = [np.median(by_mass[m]) if stat == "median" else np.mean(by_mass[m])
              for m in masses]
    sd = [np.std(by_mass[m]) for m in masses]
    return masses, center, sd, by_mass


In [ ]:
# ── Figure: P(0.075 mm) – dry vs. wet paired by specimen mass ───────────────
# All dry bars: tab:blue  |  All wet bars: tab:orange
# Lowest mass on the left, highest on the right.

MASS_PAIRS = [
    # (mass_g,  dry_keys,              wet_keys)
    (0.5,  ["0.5A", "0.5B"],      ["0.5W1","0.5W2","0.5W3"]),
    (2,    ["2A",   "2B"],        ["2W1",  "2W2",  "2W3"]),
    (10,   ["10A",  "10B"],       ["10W1", "10W2", "10W3"]),
    (50,   ["50A",  "50B"],       ["50W1", "50W2"]),
    (100,  None,                  ["100W1","100W2"]),
]

COL_DRY = "tab:blue"
COL_WET = "tab:orange"
WIDTH   = 0.35
pos     = list(range(len(MASS_PAIRS)))

fig, ax = plt.subplots()
legend_added = {"Dry sieve": False, "Wet sieve": False}

for i, (mass, dry_k, wet_k) in enumerate(MASS_PAIRS):
    xc = pos[i]

    if dry_k:
        vals = [p075(k) for k in dry_k]
        med  = np.median(vals)
        err  = [[med - min(vals)], [max(vals) - med]]
        lbl  = "Dry sieve" if not legend_added["Dry sieve"] else "_"
        legend_added["Dry sieve"] = True
        ax.bar(xc - WIDTH/2, med, width=WIDTH, color=COL_DRY,
               alpha=0.85, label=lbl, zorder=3)
        ax.errorbar(xc - WIDTH/2, med, yerr=err,
                    fmt="none", color="black", capsize=4, lw=1.2, zorder=4)

    if wet_k:
        vals = [p075(k) for k in wet_k]
        med  = np.median(vals)
        err  = [[med - min(vals)], [max(vals) - med]]
        lbl  = "Wet sieve" if not legend_added["Wet sieve"] else "_"
        legend_added["Wet sieve"] = True
        x_wet = xc + (WIDTH/2 if dry_k else 0)
        ax.bar(x_wet, med, width=WIDTH, color=COL_WET,
               alpha=0.85, label=lbl, zorder=3)
        ax.errorbar(x_wet, med, yerr=err,
                    fmt="none", color="black", capsize=4, lw=1.2, zorder=4)

ax.set_xticks(pos)
ax.set_xticklabels([f"{m:g} g" for m, *_ in MASS_PAIRS])
ax.set_ylabel("Percent passing 0.075 mm [%]")
ax.set_xlabel("Specimen mass [g]")
ax.set_ylim(0, 100)
ax.legend(frameon=False)
ax.grid(axis="y", lw=0.35)
plt.tight_layout()
savefig(fig, "fig_3_11_p075_dry_vs_wet")
plt.show()


In [ ]:
DRY_KEYS = ["50A", "50B", "10A", "10B", "2A", "2B", "0.5A", "0.5B"]
WET_LARGE = ["100W1", "100W2", "50W1", "50W2", "10W1", "10W2", "10W3"]
WET_SMALL = ["2W1", "2W2", "2W3", "0.5W1", "0.5W2", "0.5W3"]
WET_ALL = WET_LARGE + WET_SMALL

# ── Figure: D50 vs. specimen mass ────────────────────────────────────────────
fig, ax = plt.subplots()

def plot_d50_scatter(keys, label, marker, color):
    xs_all, ys_all = [], []
    for k in keys:
        d, p, _ = sieve_curve(k)
        d50 = D_at_pct(d, p, 50)
        xs_all.append(parse_specimen_mass(k))
        ys_all.append(d50)
    xs_all = np.array(xs_all, float)
    ys_all = np.array(ys_all, float)

    rng    = np.random.default_rng(0)
    xs_jit = 10 ** (np.log10(xs_all) + rng.normal(0, 0.02, len(xs_all)))
    ax.scatter(xs_jit, ys_all, marker=marker, color=color, s=55, alpha=0.85, zorder=3)

    by_mass = {}
    for x_, y_ in zip(xs_all, ys_all):
        by_mass.setdefault(x_, []).append(y_)
    masses_s = sorted(by_mass)
    medians  = [np.median(by_mass[m]) for m in masses_s]
    ax.plot(masses_s, medians, marker=marker, color=color, lw=1.8, ms=6,
            linestyle="--", label=label, zorder=4)

plot_d50_scatter(DRY_KEYS, "Dry sieve",  "o", "tab:blue")
plot_d50_scatter(WET_ALL,  "Wet sieve",  "s", "tab:orange")

ax.set_xscale("log")
unique_masses = sorted({parse_specimen_mass(k) for k in DRY_KEYS + WET_ALL})
ax.set_xticks(unique_masses)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlabel("Specimen mass [g]")
ax.set_ylabel("D50 [mm]")
ax.legend(frameon=False)
plt.tight_layout()
savefig(fig, "fig_3_12_D50_vs_mass")
plt.show()


### 3.11 Mars Reference PSD Background


In [ ]:
# ── Reference background PSD plot (Section 3.10) ───────────────────────────
# Identical structure to plot_reference_band (Section 3.8):
#   same figsize, setup_psd_ax, apply_psd_overlay, legend style.
# Difference: no min–max band, no median line; full envelope legend labels.

def plot_mars_reference(xlim=None, ylim=None,
                        show_overlay=None, show_mgs1=None):
    cfg  = PLOT_CFG
    xlim = xlim or cfg["xlim_full"]
    ylim = ylim or cfg["ylim"]
    show_overlay = cfg["show_overlay"] if show_overlay is None else show_overlay
    show_mgs1    = True               if show_mgs1    is None else show_mgs1

    fig, ax = plt.subplots(figsize=cfg.get("figsize", plt.rcParams["figure.figsize"]))

    # Envelopes with full reference labels (only difference from a plain plot)
    add_mars_envelopes(ax, show=True, full_labels=True)

    # Standard axis setup — same as every other PSD plot
    setup_psd_ax(ax, xlim=xlim, ylim=ylim,
                 xlabel="Grain size (mm, log scale)",
                 ylabel="Cumulative percent passing (%)")

    # MGS-1 line + annotated markers
    add_mgs1(ax, show=show_mgs1)

    # Overlay (USCS header + sieve ticks) — same helper, same position
    if show_overlay:
        apply_psd_overlay(ax)

    ax.legend(frameon=False, loc=cfg["legend_loc"])
    plt.tight_layout()
    return fig, ax

fig_ref, ax_ref = plot_mars_reference()
savefig(fig_ref, "figure_2_1_mars_reference_psd")
plt.show()


### 3.12 Dry vs Wet Median Sieve Curves


In [ ]:
# Figure 3.10 - Median dry and wet-prepared sieve curves.
ALL_GROUPS = {
    "50 g dry" : ["50A",  "50B"],
    "10 g dry" : ["10A",  "10B"],
    "2 g dry"  : ["2A",   "2B"],
    "0.5 g dry": ["0.5A", "0.5B"],
    "100 g wet": ["100W1","100W2"],
    "50 g wet" : ["50W1", "50W2"],
    "10 g wet" : ["10W1", "10W2","10W3"],
    "2 g wet"  : ["2W1",  "2W2", "2W3"],
    "0.5 g wet": ["0.5W1","0.5W2","0.5W3"],
}

fig, ax = plot_sieve_group_medians(
    ALL_GROUPS,
    band=None,
    xlim=SIEVE_XLIM,
)

# Dry = dashed, wet = solid for this comparison plot.
for line in ax.get_lines():
    label = (line.get_label() or "").lower()
    if "dry" in label:
        line.set_linestyle("--")
    elif "wet" in label:
        line.set_linestyle("-")

ax.legend(frameon=False, loc="lower right", fontsize=11)
savefig(fig, "fig_3_10_dry_vs_wet_medians")
plt.show()


### 3.13 PSD Method Sensitivity


In [ ]:
# ── Figure: RMS deviation from reference – method sensitivity summary ─────────
hydro_grid = np.logspace(np.log10(0.001), np.log10(0.07), 80)

# ── Reference curves ─────────────────────────────────────────────────────────
y_ref_dry = np.median(
    np.vstack([sieve_curve(k)[1] for k in ["50A","50B"]]), axis=0)

y_ref_wet = np.median(
    np.vstack([sieve_curve(k)[1] for k in ["100W1","100W2"]]), axis=0)

y_ref_bulk_h = np.nanmean(
    np.vstack([interp_to_grid(*hydro_curve(k), hydro_grid) for k in ["50A","50B"]]),
    axis=0)

y_ref_fines_h = interp_to_grid(*hydro_curve("F50", src=HYDRO_FINES), hydro_grid)

# ── RMS helper functions ──────────────────────────────────────────────────────
def rms_sieve(keys, ref):
    vals = []
    for k in keys:
        _, y, _ = sieve_curve(k)
        vals.append(np.sqrt(np.mean((y - ref)**2)))
    return float(np.mean(vals))

def rms_hydro(keys, ref, grid, src=None):
    src = HYDRO_WITH if src is None else src
    vals = []
    for k in keys:
        y = interp_to_grid(*hydro_curve(k, src=src), grid)
        diff = y - ref
        m    = np.isfinite(diff)
        if m.sum() > 5:
            vals.append(np.sqrt(np.nanmean(diff[m]**2)))
    return float(np.mean(vals)) if vals else np.nan

# ── Groups and RMS per mass ───────────────────────────────────────────────────
dry_grps   = {0.5:["0.5A","0.5B"], 2:["2A","2B"],
              10:["10A","10B"],     50:["50A","50B"]}

wet_grps   = {0.5:["0.5W1","0.5W2","0.5W3"], 2:["2W1","2W2","2W3"],
              10:["10W1","10W2","10W3"],       50:["50W1","50W2"],
              100:["100W1","100W2"]}

bulk_h_grps= {0.5:["0.5A","0.5B"], 2:["2A","2B"],
              10:["10A","10B"],     50:["50A","50B"]}

fines_h_grps = {
    0.5 : ["F0.5"],   # bulk-equivalent mass on x-axis
    2   : ["F2"],
    10  : ["F10"],
    50  : ["F50"],
}

m_dry   = sorted(dry_grps)
m_wet   = sorted(wet_grps)
m_bh    = sorted(bulk_h_grps)
m_fh    = sorted(fines_h_grps)

rms_dry  = [rms_sieve(dry_grps[m],       y_ref_dry)    for m in m_dry]
rms_wet  = [rms_sieve(wet_grps[m],       y_ref_wet)    for m in m_wet]
rms_bh   = [rms_hydro(bulk_h_grps[m],   y_ref_bulk_h,  hydro_grid)              for m in m_bh]
rms_fh   = [rms_hydro(fines_h_grps[m],  y_ref_fines_h, hydro_grid,
                       src=HYDRO_FINES)  for m in m_fh]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots()

ax.plot(m_dry, rms_dry,  "o-",  lw=2, ms=7, color="tab:blue",
        label="Dry sieve")
ax.plot(m_wet, rms_wet,  "s-",  lw=2, ms=7, color="tab:orange",
        label="Wet-prepared sieve")
ax.plot(m_bh,  rms_bh,   "^-",  lw=2, ms=7, color="tab:green",
        label="Hydrometer – bulk")
ax.plot(m_fh,  rms_fh,   "v--", lw=2, ms=7, color="tab:red",
        label="Hydrometer – fines-only")
plt.axhline(y=0, color='black', linestyle='--', linewidth=1, label="Reference")

ax.set_xscale("log")
all_masses = sorted(set(m_dry + m_wet + m_bh + m_fh))
ax.set_xticks(all_masses)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlabel("Specimen mass [g]  (bulk-equivalent for fines-only)")
ax.set_ylabel("RMS deviation from reference [%-points]")
ax.set_ylim(-3, 103)
ax.legend(frameon=False)
ax.grid(True, which="both", alpha=0.35)
plt.tight_layout()
savefig(fig, "fig_4_2_rms_method_sensitivity")
plt.show()

# Print summary table
print("RMS summary (%-points):")
print(f"{'Mass [g]':>10} {'Dry':>8} {'Wet':>8} {'Bulk H':>8} {'Fines H':>9}")
for m in all_masses:
    d = f"{dry_grps.get(m) and rms_dry[m_dry.index(m)]:.2f}" if m in m_dry else "—"
    w = f"{rms_wet[m_wet.index(m)]:.2f}" if m in m_wet else "—"
    b = f"{rms_bh[m_bh.index(m)]:.2f}"   if m in m_bh  else "—"
    f_ = f"{rms_fh[m_fh.index(m)]:.2f}"  if m in m_fh  else "—"
    print(f"{m:>10g} {d:>8} {w:>8} {b:>8} {f_:>9}")


---
## 4. Loose and Tapped Bulk Density


### 4.1 Input Data


In [ ]:
YLIM_RHO = (1, 2)
YLIM_DELTA = None

loose_raw = pd.read_csv(DATA_DIR / "loose_tapped_raw.csv")
REPLICATES = {}
value_cols = ["mass_loose_g", "V_loose_mL", "mass_tap_g", "V_tap_500", "V_tap_750", "V_tap_1250"]
for replicate, sub in loose_raw.groupby("replicate", sort=False):
    first = sub.iloc[0]
    rep = {
        "nominal_g": float(first["nominal_g"]),
        "V_cyl_mL": float(first["V_cyl_mL"]),
        "D_cyl_mm": float(first["D_cyl_mm"]),
        "m_cyl_g": float(first["m_cyl_g"]),
        "note": str(replicate),
    }
    for col in value_cols:
        rep[col] = sub[col].dropna().astype(float).tolist()
    REPLICATES[replicate] = rep

print(f"Loaded loose/tapped replicates: {len(REPLICATES)}")


### 4.2 Calculations and Summary Statistics


In [ ]:
# ── Calculations per replicate (loose + tapped) ──────────────────────────────
def nan_mean(seq):
    arr = np.array(seq, dtype=float)
    return np.nanmean(arr) if np.any(np.isfinite(arr)) else np.nan

def nan_std(seq, ddof=1):
    arr = np.array(seq, dtype=float)
    if np.isfinite(arr).sum() < 2:
        return np.nan
    return np.nanstd(arr, ddof=ddof)

def replicate_metrics(rep):
    mc = rep["m_cyl_g"]
    m_loose_net = np.array(rep["mass_loose_g"], float) - mc
    V_loose     = np.array(rep["V_loose_mL"],   float)
    m_tap_net   = np.array(rep["mass_tap_g"],   float) - mc
    V500  = np.array(rep["V_tap_500"],  float)
    V750  = np.array(rep["V_tap_750"],  float)
    V1250 = np.array(rep["V_tap_1250"], float)

    with np.errstate(divide="ignore", invalid="ignore"):
        rho_loose_i = m_loose_net / V_loose

    mbar = nan_mean(m_loose_net)
    Vbar = nan_mean(V_loose)
    rho_loose = mbar / Vbar if np.isfinite(mbar) and np.isfinite(Vbar) and Vbar > 0 else np.nan

    V500m, V750m, V1250m = nan_mean(V500), nan_mean(V750), nan_mean(V1250)
    V_tap_stable = np.nanmin([V500m, V750m, V1250m])
    stabilized = (abs(V1250m - V750m) / V750m < 0.02) if (np.isfinite(V750m) and V750m > 0) else False

    mbar_t = nan_mean(m_tap_net)
    rho_tap = mbar_t / V_tap_stable if np.isfinite(mbar_t) and np.isfinite(V_tap_stable) and V_tap_stable > 0 else np.nan

    rho_500  = mbar_t / V500m  if np.isfinite(V500m)  and V500m  > 0 else np.nan
    rho_750  = mbar_t / V750m  if np.isfinite(V750m)  and V750m  > 0 else np.nan
    rho_1250 = mbar_t / V1250m if np.isfinite(V1250m) and V1250m > 0 else np.nan

    HR = rho_tap / rho_loose if np.isfinite(rho_tap) and np.isfinite(rho_loose) and rho_loose > 0 else np.nan
    CI = (rho_tap - rho_loose) / rho_tap * 100 if np.isfinite(HR) else np.nan

    return dict(
        nominal_g = rep["nominal_g"], V_cyl_mL = rep["V_cyl_mL"],
        D_cyl_mm = rep["D_cyl_mm"], note = rep["note"],
        n_loose = int(np.isfinite(V_loose).sum()),
        m_loose_mean = mbar, m_loose_sd = nan_std(m_loose_net),
        V_loose_mean = Vbar, V_loose_sd = nan_std(V_loose),
        rho_loose = rho_loose,
        rho_loose_cv = (np.nanstd(rho_loose_i, ddof=1)/np.nanmean(rho_loose_i)*100
                        if np.isfinite(rho_loose_i).sum() > 1 else np.nan),
        gamma_d_min = rho_loose * G if np.isfinite(rho_loose) else np.nan,
        m_tap_mean = mbar_t,
        V_tap_500_mean = V500m, V_tap_750_mean = V750m, V_tap_1250_mean = V1250m,
        V_tap_stable = V_tap_stable, stabilized = stabilized,
        rho_tap_500 = rho_500, rho_tap_750 = rho_750, rho_tap_1250 = rho_1250,
        rho_tap = rho_tap,
        gamma_d_max = rho_tap * G if np.isfinite(rho_tap) else np.nan,
        HR = HR, CI = CI,
    )

records = {rid: replicate_metrics(r) for rid, r in REPLICATES.items()}
df_lt = pd.DataFrame(records).T
df_lt.index.name = "replicate"
df_lt.reset_index(inplace=True)

# ── NEW: Void ratio calculations (after supervisor comment) ──────────────────
# e = rho_s / rho_d - 1
# Uses particle density from pyknometer Section 2 (ref_rho_pyk) by default.
try:
    RHO_S_LT = float(ref_rho_pyk)
    print(f"Using rho_s from Section 2 pyknometer: {RHO_S_LT:.4f} g/cm³")
except NameError:
    RHO_S_LT = 2.987  # fallback to the value used in Section 3 hydrometer
    print(f"ref_rho_pyk not available, using fallback rho_s = {RHO_S_LT} g/cm³")

for c_in, c_out in [("rho_loose", "e_max"), ("rho_tap", "e_min")]:
    df_lt[c_out] = RHO_S_LT / pd.to_numeric(df_lt[c_in], errors="coerce") - 1.0

print(f"\nVoid ratio columns added: e_max (loose), e_min (tapped).")
df_lt[["replicate","nominal_g","V_cyl_mL","rho_loose","rho_tap","e_max","e_min"]].round(3)


### 4.3 Tables


#### Table 1: Full Replicate Table


In [ ]:
# ── Table 1: Full replicate table ────────────────────────────────────────────
cols_t1 = [
    "replicate", "nominal_g", "V_cyl_mL", "D_cyl_mm", "n_loose",
    "rho_loose", "rho_tap", "gamma_d_min", "gamma_d_max",
    "e_max", "e_min",
    "HR", "CI", "rho_loose_cv", "stabilized", "note",
]
table1 = df_lt[cols_t1].copy()
num_cols = ["rho_loose","rho_tap","gamma_d_min","gamma_d_max",
            "e_max","e_min","HR","CI","rho_loose_cv"]
for c in num_cols:
    table1[c] = pd.to_numeric(table1[c], errors="coerce").round(3)
table1.rename(columns={
    "nominal_g": "Nominal mass [g]",
    "V_cyl_mL" : "V_cyl [mL]",
    "D_cyl_mm" : "D_cyl [mm]",
    "rho_loose": "ρ_loose [g/cm³]",
    "rho_tap"  : "ρ_tap [g/cm³]",
    "gamma_d_min" : "γd,min [kN/m³]",
    "gamma_d_max" : "γd,max [kN/m³]",
    "e_max"    : "e_max (loose) [-]",
    "e_min"    : "e_min (tapped) [-]",
    "rho_loose_cv": "CV(ρ_loose) [%]",
    "stabilized"  : "Tap stable?",
    "note"        : "Note",
}, inplace=True)
table1


#### Table 2: Group Summary Table


In [ ]:
# ── Table 2: Group summary table (median +/- SD including void ratio) ────────────
for c in ["rho_loose","rho_tap","gamma_d_min","gamma_d_max","e_max","e_min","nominal_g"]:
    df_lt[c] = pd.to_numeric(df_lt[c], errors="coerce")

grouped = (df_lt.groupby("nominal_g", as_index=False)
                 .agg(n=("replicate","count"),
                      med_loose=("rho_loose","median"), sd_loose=("rho_loose","std"),
                      med_tap  =("rho_tap","median"),   sd_tap  =("rho_tap","std"),
                      med_gmin =("gamma_d_min","median"), sd_gmin=("gamma_d_min","std"),
                      med_gmax =("gamma_d_max","median"), sd_gmax=("gamma_d_max","std"),
                      med_emax =("e_max","median"),       sd_emax=("e_max","std"),
                      med_emin =("e_min","median"),       sd_emin=("e_min","std")))

m_ref = grouped["nominal_g"].max()
ref_loose = grouped.loc[grouped["nominal_g"] == m_ref, "med_loose"].iloc[0]
ref_tap   = grouped.loc[grouped["nominal_g"] == m_ref, "med_tap"  ].iloc[0]
ref_emax  = grouped.loc[grouped["nominal_g"] == m_ref, "med_emax" ].iloc[0]
ref_emin  = grouped.loc[grouped["nominal_g"] == m_ref, "med_emin" ].iloc[0]
grouped["d_loose"] = grouped["med_loose"] - ref_loose
grouped["d_tap"]   = grouped["med_tap"]   - ref_tap
grouped["d_emax"]  = grouped["med_emax"]  - ref_emax
grouped["d_emin"]  = grouped["med_emin"]  - ref_emin

def fmt_pm(m, s, prec=3):
    if pd.isna(m): return ""
    if pd.isna(s): return f"{m:.{prec}f}"
    return f"{m:.{prec}f} ± {s:.{prec}f}"

master = pd.DataFrame({
    "Massegruppe" : [f"{int(m) if float(m).is_integer() else m:g} g" for m in grouped["nominal_g"]],
    "n"           : grouped["n"].astype(int),
    "ρ_loose [g/cm³]" : [fmt_pm(m, s, 3) for m, s in zip(grouped["med_loose"], grouped["sd_loose"])],
    "ρ_tap [g/cm³]"   : [fmt_pm(m, s, 3) for m, s in zip(grouped["med_tap"],   grouped["sd_tap"])],
    "γd,min [kN/m³]"  : [fmt_pm(m, s, 2) for m, s in zip(grouped["med_gmin"],  grouped["sd_gmin"])],
    "γd,max [kN/m³]"  : [fmt_pm(m, s, 2) for m, s in zip(grouped["med_gmax"],  grouped["sd_gmax"])],
    "e_max (loose) [-]" : [fmt_pm(m, s, 3) for m, s in zip(grouped["med_emax"], grouped["sd_emax"])],
    "e_min (tapped) [-]": [fmt_pm(m, s, 3) for m, s in zip(grouped["med_emin"], grouped["sd_emin"])],
})
master


### 4.4 Thesis Figures


In [ ]:
LT_PLOT_OPTS = {
    "show_replicates": True,
    "show_median": True,
    "show_sd": False,
    "show_ref_lines": True,
    "ylim_rho": (0.8, 1.80),
    "ylim_e": (0.8, 1.80),
    "ylim_tap_volnorm": (0.65, 1.05),
    "figsize_compare": (14, 6.5),
    "figsize_tap": (10, 6.5),
    "replicate_ls": {0: "-", 1: (0, (6, 2)), 2: (0, (1, 1.5))},
}
x = grouped["nominal_g"].to_numpy()

# ── Fig 4.3 – ρ_d og void ratio side-om-side ────────────────────────────────
# Arver toggles fra LT_PLOT_OPTS slik at figuren ligger konsekvent med Fig 4.1/4.2.
fig3_lt, (ax1, ax2) = plt.subplots(1, 2, figsize=LT_PLOT_OPTS["figsize_compare"])

# --- Replicateer (scatter) ---
if LT_PLOT_OPTS["show_replicates"]:
    np.random.seed(2)
    for _, row in df_lt.iterrows():
        m  = row["nominal_g"]
        col = mass_color(m); mk = mass_marker(m)
        jx = m * np.random.uniform(0.94, 1.06)
        # ρ_d
        ax1.plot(jx, row["rho_loose"], marker=mk, ms=6, color=col,
                 markerfacecolor=col, markeredgecolor=col,
                 alpha=0.38, linestyle="none", zorder=1.5)
        ax1.plot(jx, row["rho_tap"], marker=mk, ms=6, color=col,
                 markerfacecolor="white", markeredgecolor=col,
                 alpha=0.55, linestyle="none", zorder=1.5)
        # e
        ax2.plot(jx, row["e_max"], marker=mk, ms=6, color=col,
                 markerfacecolor=col, markeredgecolor=col,
                 alpha=0.38, linestyle="none", zorder=1.5)
        ax2.plot(jx, row["e_min"], marker=mk, ms=6, color=col,
                 markerfacecolor="white", markeredgecolor=col,
                 alpha=0.55, linestyle="none", zorder=1.5)

# --- Gruppemedian + SD ---
_show_med = LT_PLOT_OPTS["show_median"]
_show_sd  = LT_PLOT_OPTS["show_sd"]
if _show_med or _show_sd:
    for xi, yl, yl_err, yt, yt_err, ye_mx, ye_mx_sd, ye_mn, ye_mn_sd in zip(
            x,
            grouped["med_loose"], grouped["sd_loose"],
            grouped["med_tap"],   grouped["sd_tap"],
            grouped["med_emax"],  grouped["sd_emax"],
            grouped["med_emin"],  grouped["sd_emin"]):
        col = mass_color(xi); mk = mass_marker(xi)
        ms_val = 8 if _show_med else 0
        el = yl_err if _show_sd else 0
        et = yt_err if _show_sd else 0
        ex = ye_mx_sd if _show_sd else 0
        en = ye_mn_sd if _show_sd else 0
        # ρ_d subplot
        ax1.errorbar(xi, yl, yerr=el, fmt=mk, ms=ms_val, color=col,
                     markerfacecolor=col, markeredgecolor=col,
                     capsize=4, lw=0, zorder=3)
        ax1.errorbar(xi, yt, yerr=et, fmt=mk, ms=ms_val, color=col,
                     markerfacecolor="white", markeredgecolor=col,
                     capsize=4, lw=0, zorder=3)
        # e subplot
        ax2.errorbar(xi, ye_mx, yerr=ex, fmt=mk, ms=ms_val, color=col,
                     markerfacecolor=col, markeredgecolor=col,
                     capsize=4, lw=0, zorder=3)
        ax2.errorbar(xi, ye_mn, yerr=en, fmt=mk, ms=ms_val, color=col,
                     markerfacecolor="white", markeredgecolor=col,
                     capsize=4, lw=0, zorder=3)
    if _show_med:
        ax1.plot(x, grouped["med_loose"], color="0.4", ls="-",  lw=1.2, zorder=1)
        ax1.plot(x, grouped["med_tap"],   color="0.4", ls="--", lw=1.2, zorder=1)
        ax2.plot(x, grouped["med_emax"],  color="0.4", ls="-",  lw=1.2, zorder=1)
        ax2.plot(x, grouped["med_emin"],  color="0.4", ls="--", lw=1.2, zorder=1)

for axi, ylab, ytitle in [(ax1, r"$\rho_d$ [g/cm³]", "Dry bulk density"),
                           (ax2, r"Void ratio, $e$ [-]", "Void ratio")]:
    axi.set_xscale("log")
    axi.set_xticks(sorted(x))
    axi.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    axi.set_xlabel("Nominal specimen mass [g]")
    axi.set_ylabel(ylab)
    axi.set_title(ytitle)

if LT_PLOT_OPTS["ylim_rho"] is not None:
    ax1.set_ylim(LT_PLOT_OPTS["ylim_rho"])
if LT_PLOT_OPTS["ylim_e"] is not None:
    ax2.set_ylim(LT_PLOT_OPTS["ylim_e"])

mass_handles = [Line2D([0],[0], marker=mass_marker(m), color=mass_color(m),
                        lw=0, markerfacecolor=mass_color(m), markersize=8,
                        label=f"{m:g} g") for m in sorted(set(x))]
state_handles = [
    Line2D([0],[0], marker="o", lw=0, color="0.3", markerfacecolor="0.3",
           markersize=8, label="Loose / $e_{max}$"),
    Line2D([0],[0], marker="o", lw=0, color="0.3", markerfacecolor="white",
           markersize=8, label="Tapped / $e_{min}$"),
]
fig3_lt.legend(handles=mass_handles + state_handles,
                frameon=False, ncol=len(mass_handles)+2,
                loc="upper center", bbox_to_anchor=(0.5, 1.04))

plt.tight_layout()
savefig(fig3_lt, "fig_3_18_density_void_ratio")
plt.show()


In [ ]:
# ── Fig 4.4 – Tapping curve: V(N)/V_loose vs number of taps (B/W readable) ───────────
# B/W design: colour+marker per mass, linestyle per replicate (solid/dashed/dotted).
# This keeps each combination readable in both colour and black-and-white.
fig4_lt, ax = plt.subplots(figsize=LT_PLOT_OPTS["figsize_tap"])
taps = np.array([0, 500, 750, 1250])

CYL_LABEL = {100: "100 mL (diam. 25 mm)", 25: "25 mL (diam. 19 mm)", 10: "10 mL (diam. 15 mm)"}

rep_counter = {}   # assign per-mass replicate index
for _, row in df_lt.iterrows():
    V_loose_mean = float(row["V_loose_mean"]) if np.isfinite(row["V_loose_mean"]) else np.nan
    V_tap = [V_loose_mean, row["V_tap_500_mean"], row["V_tap_750_mean"], row["V_tap_1250_mean"]]
    V_tap = np.array([float(v) if np.isfinite(v) else np.nan for v in V_tap])
    if not np.isfinite(V_loose_mean) or V_loose_mean <= 0:
        continue
    ratio = V_tap / V_loose_mean
    m    = float(row["nominal_g"])
    col  = mass_color(m); mk = mass_marker(m)
    # Index replicates within each mass -> 0,1,2 for linestyle lookup
    rep_idx = rep_counter.get(m, 0)
    rep_counter[m] = rep_idx + 1
    ls  = LT_PLOT_OPTS["replicate_ls"].get(rep_idx, "-")
    ax.plot(taps, ratio, marker=mk, color=col, ls=ls,
            markerfacecolor=col, markeredgecolor=col, markeredgewidth=1.1,
            linewidth=1.5, alpha=0.9,
            label=f"{row['replicate']} ({m:g} g, {int(row['V_cyl_mL'])} mL)")

ax.set_xlabel("Number of taps (Retsch AS 200, 3 mm amplitude)")
ax.set_ylabel(r"$V(N)/V_{loose}$ [–]")
if LT_PLOT_OPTS["ylim_tap_volnorm"] is not None:
    ax.set_ylim(LT_PLOT_OPTS["ylim_tap_volnorm"])

# Two-part legend: mass (colour/marker) + replicate (linestyle)
mass_handles = [Line2D([0],[0], marker=mass_marker(mm), color=mass_color(mm),
                        lw=0, markerfacecolor=mass_color(mm), markersize=6.5,
                        label=f"{mm:g} g")
                for mm in sorted(df_lt["nominal_g"].unique())]
rep_handles = [Line2D([0],[0], color="0.3",
                       ls=LT_PLOT_OPTS["replicate_ls"].get(i, "-"),
                       lw=1.5, label=f"Replicate {i+1}") for i in range(3)]
ax.legend(handles=mass_handles + rep_handles,
          frameon=False, fontsize=11, ncol=2, loc="upper right")
plt.tight_layout()
savefig(fig4_lt, "fig_3_19_tapping_normalized")
plt.show()
